In [ ]:
import pandas as pd
import random


def generate_client_ids(n=10):
    """Generate n random 10-digit client ID strings."""
    ids = []
    for _ in range(n):
        id_code = "".join(str(random.randint(0, 9)) for _ in range(10))
        ids.append(id_code)
    return ids


n = 10

df = pd.DataFrame({"document_PatientDurableKey": generate_client_ids(n)})

df.to_csv("treatment_docs.csv", index=False)

print("CSV file 'treatment_docs.csv' created!")
print(df.head())

In [ ]:
import numpy as np
import os
import sys
import shutil

random_seed_value = 42

np.random.seed(random_seed_value)

random.seed(random_seed_value)

In [ ]:
print("Current Working Directory:", os.getcwd())

print("Python Path:", sys.path)

In [ ]:
clear_previous_outputs = True

if clear_previous_outputs:
    shutil.rmtree("new_project", ignore_errors=True)
    shutil.rmtree("new_project_ipw", ignore_errors=True)
    shutil.rmtree("treatment_doc_extract", ignore_errors=True)

In [ ]:
current_dir = os.getcwd()

path_to_medcat_model_pack = os.path.abspath(
    os.path.join(
        current_dir,
        "..",
        "..",
        "medcat_models",
        "medcat_model_pack_422d1d38fc58f158.zip",
    )
)

path_to_snomed_ct_file = os.path.abspath(
    os.path.join(
        current_dir,
        "..",
        "..",
        "snomed",
        "SnomedCT_InternationalRF2_PRODUCTION_20231101T120000Z",
        "SnomedCT_InternationalRF2_PRODUCTION_20231101T120000Z",
        "Full",
        "Terminology",
        "sct2_StatedRelationship_Full_INT_20231101.txt",
    )
)

path_to_gloabl_files = "../../"

additional_path_to_pat2vec = os.path.abspath(
    os.path.join(path_to_gloabl_files, "pat2vec")
)

absolute_path = os.path.abspath(os.path.join(current_dir, path_to_gloabl_files))

print(path_to_medcat_model_pack)
print(path_to_snomed_ct_file)
print(path_to_gloabl_files)
print(additional_path_to_pat2vec)

In [ ]:
sys.path.insert(0, path_to_gloabl_files)
sys.path.insert(0, additional_path_to_pat2vec)

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

grandparent_dir = os.path.dirname(parent_dir)
sys.path.append(grandparent_dir)

### Set up logger

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()

### Database Backend Setup

The `pat2vec` pipeline now supports a database backend for handling large datasets and ensuring data persistence.

For this example, we will set up an **ephemeral SQLite database**. This allows us to run the pipeline with a clean state, storing all intermediate outputs (raw data, annotations, features) in a local file that is reset on each run.

In [ ]:
import os

PROJ_NAME = "new_project"
DB_FILENAME = "temp_test_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    os.remove(DB_PATH)
    print(f"Removed old database file: {DB_PATH}")
except FileNotFoundError:
    print(f"No old database file to remove. A new one will be created at: {DB_PATH}")

db_connection_string = f"sqlite:///{DB_PATH}"

print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class
from datetime import datetime
from tqdm import tqdm
from pat2vec.util.post_processing import extract_datetime_to_column
from dateutil.relativedelta import relativedelta
import pandas as pd
from typing import Dict, List, Optional, Union

main_options_dict = {
    "demo": False,
    "bmi": False,
    "bloods": False,
    "drugs": False,
    "diagnostics": False,
    "core_02": False,
    "bed": False,
    "vte_status": False,
    "hosp_site": False,
    "core_resus": False,
    "news": False,
    "smoking": False,
    "annotations": False,
    "annotations_mrc": False,
    "negated_presence_annotations": False,
    "appointments": False,
    "annotations_reports": False,
    "textual_obs": False,
    "covid": False,
    "epic_encounters": True,
    # Note: epic_clinical_notes is only available via annotations
    "epic_clinical_notes_annotations": True,
    "epic_medical_history_annotations": True,
    "epic_orders_annotations": True,
    "epic_lab_results": True,
    "epic_patients": True,
    "epic_imaging_reports_annotations": True,
    "epic_clinical_notes_appointments": True,
}

annot_filter_arguments = {
    "acc": 0.8,
    "types": [
        "qualifier value",
        "procedure",
        "substance",
        "finding",
        "environment",
        "disorder",
        "observable entity",
    ],
    "Time_Value": ["Recent", "Past"],
    "Time_Confidence": 0.8,
    "Presence_Value": ["True"],
    "Presence_Confidence": 0.8,
    "Subject_Value": ["Patient"],
    "Subject_Confidence": 0.8,
}

epr_docs_term_regex: Optional[Union[str, None]] = None
mct_docs_term_regex: Optional[Union[str, None]] = None

bloods_filter_term_list: Optional[Union[List[str], None]] = None

mct_docs_document_type_filter_list: Optional[Union[List[str], None]] = None
epr_docs_document_type_filter_list: Optional[Union[List[str], None]] = None

data_type_filter_dict: Dict[str, any] = {
    "filter_term_lists": {
        "epr_docs": epr_docs_document_type_filter_list,
        "mct_docs": mct_docs_document_type_filter_list,
        "bloods": bloods_filter_term_list,
    },
    "epr_docs_term_regex": epr_docs_term_regex,
    "mct_docs_term_regex": mct_docs_term_regex,
}

config_obj = config_class(
    remote_dump=False,
    suffix="",
    treatment_doc_filename="test_files/treatment_docs.csv",
    treatment_control_ratio_n=1,
    proj_name="new_project",
    current_path_dir="",
    main_options=main_options_dict,
    start_date=(datetime(1995, 1, 1)),
    years=30,
    months=0,
    days=0,
    batch_mode=True,
    store_annot=True,
    share_sftp=True,
    multi_process=False,
    strip_list=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=datetime.now(),
    patient_id_column_name="auto",
    annot_filter_options=annot_filter_arguments,
    global_start_year=1995,
    global_start_month=1,
    global_end_year=2025,
    global_end_month=1,
    global_start_day=1,
    global_end_day=1,
    shuffle_pat_list=False,
    time_window_interval_delta=relativedelta(years=31),
    split_clinical_notes=True,
    lookback=False,
    add_icd10=False,
    add_opc4s=False,
    override_medcat_model_path=path_to_medcat_model_pack,
    data_type_filter_dict=None,
    filter_split_notes=True,
    prefetch_pat_batches=False,
    sample_treatment_docs=5,
    credentials_path="../util/credentials.py",
    storage_backend="database",
    db_connection_string=db_connection_string,
)

In [ ]:
from pat2vec.main_pat2vec import main

In [ ]:
pat2vec_obj = main(
    cogstack=True,
    use_filter=False,
    json_filter_path=None,
    random_seed_val=42,
    hostname=None,
    config_obj=config_obj,
)

View patient list

In [ ]:
pat2vec_obj.all_patient_list[0:8]

In [ ]:
pat2vec_obj.config_obj.date_list

Make pat vectors for pat 0

In [ ]:
pat2vec_obj.pat_maker(0)

In [ ]:
from pat2vec.util.post_processing import remove_file_from_paths

In [ ]:
MAX_RETRIES = 3

for i in tqdm(range(0, len(pat2vec_obj.all_patient_list))):
    retries = 0
    success = False

    while retries < MAX_RETRIES and not success:
        try:
            pat2vec_obj.pat_maker(i)
            success = True

        except KeyError as e:
            print(f"KeyError at index {i}: {e}. Retrying after removal...")
            remove_file_from_paths(pat2vec_obj.all_patient_list[i])
            retries += 1

        except Exception as e:
            print(f"Exception at index {i}: {e}. Skipping this patient...")
            break

        finally:
            pat2vec_obj.t.update(1)

    if not success:
        print(f"Failed to process index {i} after {MAX_RETRIES} retries.")

pat2vec_obj.t.close()

In [ ]:
from pat2vec.util.helper_functions import get_all_features
import os

df = get_all_features(pat2vec_obj.config_obj)

output_directory = f"{pat2vec_obj.proj_name}/output_directory"
output_file = f"{output_directory}/output_file.csv"

os.makedirs(output_directory, exist_ok=True)

df.to_csv(output_file, index=False)

print(f"Features exported to {output_file}")
print(f"Total shape: {df.shape}")

In [ ]:
df = pd.read_csv(output_file)

In [ ]:
df = extract_datetime_to_column(df)

In [ ]:
df

#### Build all document batches dataframe:

In [ ]:
from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

all_pat_list = pat2vec_obj.all_patient_list

dfd = build_merged_epr_mct_doc_df(all_pat_list, pat2vec_obj.config_obj, overwrite=True)

### Build all annotation batches dataframe:

In [ ]:
from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_annot_df

all_pat_list = pat2vec_obj.all_patient_list

dfa = build_merged_epr_mct_annot_df(
    all_pat_list, pat2vec_obj.config_obj, overwrite=True
)

dfa = pd.read_csv(dfa)

dfa

In [ ]:
dfa["annotation_batch_source"].value_counts()